# S5: Layer-Wise PEFT Insertion (Optional)

**Leaf-JEPA IRP** | Stage 5 — PEFT Adaptation Experiments


**Research Role:** Reveals which transformer layers benefit most from adaptation.
Connects Stage 4's disease-biased masking to where adaptation helps.

**Experiment:** LoRA (r=8) applied to early (0–10), middle (11–20), late (21–31), or all blocks.

## Imports & Configuration

In [ ]:
import sys
import json
from pathlib import Path
from collections import defaultdict

PROJECT_ROOT = Path("..").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import matplotlib.pyplot as plt

from stage5_peft_adaptation_experiments.config_stage5 import *
from stage5_peft_adaptation_experiments.peft_utils import (
    set_seed, get_device, train_peft, aggregate_seed_results,
    save_results
)

verify_config()
device = get_device()

SET5_DIR = PEFT_DIR / "set5"
SET5_DIR.mkdir(parents=True, exist_ok=True)

## Run Layer-Group Ablation

4 layer groups × 3 seeds = 12 runs. All use LoRA r=8.

In [ ]:
all_results = []

for group_name, block_indices in LAYER_GROUPS.items():
    print(f"\n{'='*60}")
    print(f"  LAYER GROUP: {group_name} "
          f"(blocks {block_indices[0]}–{block_indices[-1]}, "
          f"n={len(block_indices)} blocks)")
    print(f"{'='*60}")
    
    for seed in SUBSET_SEEDS:
        run_name = f"S5-LoRA-r8-layers_{group_name}-frac1.00-seed{seed}"
        print(f"\n  {run_name}")
        
        result = train_peft(
            method="lora",
            checkpoint_path=LEAF_JEPA_CHECKPOINT,
            pv_root=PV_ROOT,
            splits_dir=SPLITS_DIR,
            norm_mean=NORM_MEAN,
            norm_std=NORM_STD,
            model_name=VIT_MODEL_NAME,
            embed_dim=VIT_EMBED_DIM,
            num_classes=NUM_CLASSES,
            fraction=1.0,
            seed=seed,
            lr=BEST_LR["lora"],
            batch_size=BATCH_SIZE,
            max_epochs=MAX_EPOCHS,
            patience=EARLY_STOP_PATIENCE,
            weight_decay=WEIGHT_DECAY,
            warmup_fraction=WARMUP_FRACTION,
            use_amp=USE_AMP,
            gradient_clip=GRADIENT_CLIP,
            num_workers=NUM_WORKERS,
            class_weights_path=CLASS_WEIGHTS_PATH,
            save_dir=SET5_DIR,
            run_name=run_name,
            wandb_project=WANDB_PROJECT,
            wandb_entity=WANDB_ENTITY,
            wandb_group=WANDB_GROUPS["set5"],
            rank=8,
            target_blocks=block_indices,
        )
        result["layer_group"] = group_name
        all_results.append(result)

save_results(all_results, PEFT_DIR / "set5_layer_ablation.json")
print(f"\n✅ Layer ablation complete: {len(all_results)} runs")

## Analysis & Visualisation

In [ ]:
grouped = defaultdict(list)
for res in all_results:
    grouped[res["layer_group"]].append(res["results"]["test_macro_f1"])

print("\nLayer Ablation Results (LoRA r=8)")
print("=" * 55)

group_order = ["early", "middle", "late", "all"]
means = []
stds = []
params_list = []

for group in group_order:
    scores = grouped[group]
    m = np.mean(scores)
    s = np.std(scores)
    means.append(m)
    stds.append(s)
    
    p = [r["param_count"]["trainable"]
         for r in all_results if r["layer_group"] == group][0]
    params_list.append(p)
    
    print(f"  {group:<8} | F1: {m:.4f} ± {s:.4f} | Params: {p:,}")

## Bar Chart

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

colours = ["#3498db", "#2ecc71", "#e74c3c", "#9b59b6"]
bars = ax.bar(group_order, means, yerr=stds, capsize=5,
              color=colours, alpha=0.85)

ax.set_xlabel("Layer Group", fontsize=12)
ax.set_ylabel("Macro-F1", fontsize=12)
ax.set_title("LoRA (r=8): Performance by Layer Group", fontsize=14)
ax.grid(axis="y", alpha=0.3)

for bar, m, p in zip(bars, means, params_list):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max(stds) * 0.3,
        f"{m:.3f}\n({p:,}p)",
        ha="center", fontsize=9,
    )

plt.tight_layout()
plt.savefig(FIGURES_DIR / "set5_layer_groups.png", dpi=150)
plt.show()

## Interpretation

In [ ]:
# Identify the dominant layer group
best_group = group_order[np.argmax(means)]
late_f1 = means[group_order.index("late")]
early_f1 = means[group_order.index("early")]
all_f1 = means[group_order.index("all")]

print("\n★ KEY FINDINGS:")
print("=" * 55)

if late_f1 > early_f1 + 0.01:
    print("  Late-layer LoRA captures most of the adaptation benefit.")
    print("  → Disease-biased masking primarily improved high-level")
    print("    semantic representations (disease type, severity).")
    print()
    if abs(late_f1 - all_f1) < 0.005:
        print("  Late-only LoRA matches all-layer LoRA!")
        print("  → Practical implication: only adapt late layers for efficiency.")
elif abs(late_f1 - early_f1) < 0.01:
    print("  LoRA benefit is UNIFORM across layers.")
    print("  → Disease-biased masking improved features at ALL scales")
    print("    (low-level textures, mid-level structures, high-level semantics).")
else:
    print(f"  Highest performing group: {best_group}")
    print(f"  Early: {early_f1:.4f} | Late: {late_f1:.4f} | All: {all_f1:.4f}")

print("\n✅ Set 5 complete.")